In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

import warnings
warnings.filterwarnings('ignore')

In [10]:
df=pd.read_csv("qoute_dataset.csv")
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [11]:
df.shape

(3038, 2)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   3038 non-null   object
 1   Author  3038 non-null   object
dtypes: object(2)
memory usage: 47.6+ KB


In [13]:
quotes= df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [14]:
quotes= quotes.str.lower()

In [15]:
translator= str.maketrans('','',string.punctuation)
quotes= quotes.apply(lambda x:x.translate(translator))

In [16]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [17]:
vocal_size=8978

tok=Tokenizer(num_words=vocal_size)
tok.fit_on_texts(quotes)


In [18]:
word_index= tok.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [19]:
sequence= tok.texts_to_sequences(quotes)

In [20]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [21]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [22]:
X= []
y= []

for seq in sequence:
    for i in range(1,len(seq)):
      input_seq= seq[:i]
      output_seq= seq[i]
      X.append(input_seq)
      y.append(output_seq)

In [23]:
len(X)

85270

In [24]:
len(y)

85270

In [25]:
max_len= max(len(x) for x in X)
max_len

745

In [26]:
X_padded= pad_sequences(X,maxlen=max_len,padding='pre')

In [27]:
y=np.array(y)

In [28]:
X_padded.shape

(85270, 745)

In [29]:
y.shape

(85270,)

In [30]:
y

array([ 62,  29,  19, ...,   3, 169, 101])

In [31]:
y_onehot= to_categorical(y,num_classes=vocal_size)
y_onehot

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [32]:
embedding= 50
rnn_units= 128
rnn_model= Sequential()
rnn_model.add(Embedding(input_dim=vocal_size,output_dim=embedding,input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_units,return_sequences=True))
rnn_model.add(Dense(units=vocal_size,activation='softmax'))

In [33]:
rnn_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [34]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
lstm_model= Sequential()
lstm_model.add(Embedding(input_dim=vocal_size,output_dim=embedding,input_length=max_len))

lstm_model.add(LSTM(units=rnn_units,return_sequences=True))
lstm_model.add(Dense(units=vocal_size,activation='softmax'))

In [36]:
lstm_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [37]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [38]:
embedding= 50
rnn_units= 128
rnn_model= Sequential()
rnn_model.add(Embedding(input_dim=vocal_size,output_dim=embedding,input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_units,return_sequences=False)) # FIX: Changed to False
rnn_model.add(Dense(units=vocal_size,activation='softmax'))

rnn_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

epochs=10
batch_size= 128
history_rnn=rnn_model.fit(X_padded,y_onehot,epochs=epochs,batch_size=batch_size,validation_split=0.1)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 47s 70ms/step - accuracy: 0.0453 - loss: 6.6973 - val_accuracy: 0.0559 - val_loss: 6.5465
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 76s 65ms/step - accuracy: 0.0618 - loss: 6.5835 - val_accuracy: 0.0670 - val_loss: 6.6234
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.0680 - loss: 6.4712 - val_accuracy: 0.0732 - val_loss: 6.5557
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.0828 - loss: 6.0366 - val_accuracy: 0.0805 - val_loss: 6.5190
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.0922 - loss: 5.8822 - val_accuracy: 0.0876 - val_loss: 6.4981
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.0979 - loss: 5.7417 - val_accuracy: 0.0914 - val_loss: 6.4972
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.1034 - loss: 5.6675 - val_accuracy: 0.0929 - val_loss: 6.5019
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.1090 - loss: 5.5186 - 

In [39]:
# Redefine the LSTM model with return_sequences=False
lstm_model= Sequential()
lstm_model.add(Embedding(input_dim=vocal_size,output_dim=embedding,input_length=max_len))
lstm_model.add(LSTM(units=rnn_units,return_sequences=False)) # FIX: Changed to False
lstm_model.add(Dense(units=vocal_size,activation='softmax'))

# Recompile the model
lstm_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

epochs=100
batch_size= 128
history_lstm=lstm_model.fit(X_padded,y_onehot,epochs=epochs,batch_size=batch_size,validation_split=0.1)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 57ms/step - accuracy: 0.0388 - loss: 6.7445 - val_accuracy: 0.0425 - val_loss: 6.6826
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 36s 55ms/step - accuracy: 0.0624 - loss: 6.2942 - val_accuracy: 0.0731 - val_loss: 6.5009
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0845 - loss: 6.0255 - val_accuracy: 0.0884 - val_loss: 6.4506
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.0987 - loss: 5.8187 - val_accuracy: 0.0969 - val_loss: 6.4184
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1098 - loss: 5.6401 - val_accuracy: 0.1010 - val_loss: 6.4077
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 55ms/step - accuracy: 0.1191 - loss: 5.4683 - val_accuracy: 0.1047 - val_loss: 6.4178
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1283 - loss: 5.3108 - val_accuracy: 0.1071 - val_loss: 6.4243
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1353 - loss: 5

In [40]:
lstm_model.save('lstm_model.h5')
rnn_model.save('rnn_model.h5')

In [41]:
index_to_word= {}
for word,index in word_index.items():
  index_to_word[index]= word

In [42]:
def perdictor(model,tok,text,max_len):
  text= text.lower()
  seq= tok.texts_to_sequences([text])[0]
  seq= pad_sequences([seq],maxlen=max_len,padding='pre')

  pred= model.predict(seq,verbose=0)
  pred_index= np.argmax(pred)
  return index_to_word[pred_index]


In [43]:
seed_text="what is"
next_word= perdictor(lstm_model,tok,seed_text,max_len)
print(next_word)

an


In [47]:
def generate_text(model,tok,seed_text,n_words,max_len):
  for _ in range(n_words):
    next_word= perdictor(model,tok,seed_text,max_len)
    if next_word is "":
      break
    seed_text += " "+ next_word
  return seed_text

In [45]:
seed= "the meaning of life"
generate_text= generate_text(lstm_model,tok,seed,10,max_len)
print(generate_text)

the meaning of life is to be broken but if your dream has to


In [48]:
import pickle

with open ("tokenizer.pkl","wb") as f:
  pickle.dump(tok,f)

In [49]:
with open ("max_len.pkl","wb") as f:
  pickle.dump(max_len,f)